# Inference Demo — Heart Disease Risk Classifier

This notebook demonstrates loading the packaged model artifact (`models/heart_model.joblib` + `models/model_metadata.json`) and running inference directly -- no API server needed. It mirrors exactly what `src/heart_mlops/api/main.py`'s `/predict` endpoint does internally (same feature-column order, same `float64` cast, same threshold-based decision rule), so this is a faithful standalone demonstration of the serving path, not a different implementation of it.

See `docs/DECISIONS.md` D5.1/D5.2 for why the joblib artifact (not the MLflow registry) is what actually ships.

In [1]:
import json

import joblib
import pandas as pd

from heart_mlops.config import (
    CLEANED_DATA_PATH,
    MODEL_FEATURE_COLUMNS,
    MODELS_DIR,
    PROJECT_ROOT,
)

pipeline = joblib.load(MODELS_DIR / "heart_model.joblib")
metadata = json.loads((MODELS_DIR / "model_metadata.json").read_text())

print(f"Model: {metadata['model_name']} (registry v{metadata['model_version']})")
print(f"Operating threshold: {metadata['threshold']:.4f}")
print(f"Feature columns, in the exact order the pipeline expects: {MODEL_FEATURE_COLUMNS}")

Model: logreg (registry v2)
Operating threshold: 0.2419
Feature columns, in the exact order the pipeline expects: ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']


## Single-record inference

Using the same sample payloads the API's own container smoke test sends (`test/sample_requests/`), so this notebook and the live API are demonstrably exercising the identical model on identical input.

In [2]:
def predict_one(record: dict) -> dict:
    """Mirrors api/main.py's _features_to_row + _risk_to_response exactly."""
    row = pd.DataFrame([record])[MODEL_FEATURE_COLUMNS].astype("float64")
    risk_probability = float(pipeline.predict_proba(row)[0, 1])
    threshold = metadata["threshold"]
    prediction = int(risk_probability >= threshold)
    label = "Disease" if prediction == 1 else "No disease"
    confidence = risk_probability if prediction == 1 else 1.0 - risk_probability
    return {
        "prediction": prediction,
        "label": label,
        "confidence": round(confidence, 4),
        "risk_probability": round(risk_probability, 4),
    }


sample_dir = PROJECT_ROOT / "test" / "sample_requests"
for name in ["valid_high_risk.json", "valid_low_risk.json"]:
    record = json.loads((sample_dir / name).read_text())
    result = predict_one(record)
    print(f"{name}: {result}")

valid_high_risk.json: {'prediction': 1, 'label': 'Disease', 'confidence': 0.3336, 'risk_probability': 0.3336}
valid_low_risk.json: {'prediction': 0, 'label': 'No disease', 'confidence': 0.9068, 'risk_probability': 0.0932}


## Batch inference on real patients, with ground truth

A handful of real rows from the cleaned, combined 4-site dataset (fixed seed, so this is reproducible), run through the same pipeline in one vectorised call -- the same approach `POST /predict/batch` uses -- compared against their actual recorded diagnosis.

In [3]:
df = pd.read_csv(CLEANED_DATA_PATH)
sample = df.dropna(subset=MODEL_FEATURE_COLUMNS).sample(8, random_state=42)

X = sample[MODEL_FEATURE_COLUMNS].astype("float64")
risk_probabilities = pipeline.predict_proba(X)[:, 1]
threshold = metadata["threshold"]
predictions = (risk_probabilities >= threshold).astype(int)

results = sample[["site", "target"]].copy()
results["risk_probability"] = risk_probabilities.round(4)
results["predicted"] = predictions
results["correct"] = results["predicted"] == results["target"]
results

,site,target,risk_probability,predicted,correct
285,cleveland,1,0.8982,1,True
269,cleveland,0,0.0532,0,True
165,cleveland,0,0.6031,1,False
9,cleveland,1,0.8828,1,True
77,cleveland,0,0.2375,0,True
282,cleveland,1,0.9174,1,True
94,cleveland,0,0.0262,0,True
110,cleveland,1,0.7424,1,True


## Takeaway

This is the exact artifact and decision rule running in production (the Docker image, the Kubernetes deployment, and the live `/predict` endpoint all load this same `heart_model.joblib` and apply this same threshold) -- there is no separate "notebook version" of the model to drift out of sync with what actually serves traffic.